<a href="https://colab.research.google.com/github/hannahapel/masterthesis_tell-me-whats-mine-tell-me-whats-yours/blob/main/Appendix%2010(4)_Final%20Post%20Retrieval%20Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% CELL 1 — Install
!pip install transformers torch openpyxl pandas -q

In [ ]:
# %% CELL 2 — Upload the FULL CORPUS file
# This must be your complete Candidate Posts file (or the still-unclassified
# remainder of it), with at minimum columns: a unique id, 'title', 'selftext'.
from google.colab import files
uploaded = files.upload()  # select your full corpus CSV/export
INPUT_CSV = list(uploaded.keys())[0]
ID_COLUMN = "post_id"  # CHANGE this to match your file's actual ID column name

Saving Final_Retrieval_148_RESULTS_v1_3.xlsx to Final_Retrieval_148_RESULTS_v1_3.xlsx


In [ ]:
# %% CELL 3 — Final candidate labels, template, and calibrated thresholds (locked in)
CANDIDATE_LABELS = {
    "modality_image": "AI-generated or AI-manipulated still images, such as face swaps or synthetic faces",
    "modality_video": "AI-manipulated video, such as face swap, lip-sync, or facial and body manipulation",
    "modality_audio": "AI-generated or AI-cloned voice, such as voice cloning or speech synthesis",
    "modality_text": "AI-generated or AI-manipulated text, including text-to-speech conversion",
    "modality_multimodal": "two or more AI-manipulated modalities combined and synchronized in the same content, such as a lip-sync video with a cloned voice",
    "crime_biometric_bypass": "using AI to bypass biometric authentication, such as face or voice recognition systems",
    "crime_financial_fraud_ceo": "an AI-generated voice or video used to scam someone out of money, such as a cloned voice pretending to be a relative, boss, or trusted person asking for a money transfer",
    "crime_credential_theft": "sensitive personal or company information being leaked or exposed to AI tools, or someone using AI to fake their professional credentials, resume, or work history",
    "crime_document_forgery": "using AI to forge or falsify official documents or identification",
    "crime_political_disinfo": "AI-generated political disinformation or propaganda",
    "crime_reputational_sabotage": "using AI-manipulated content to damage a person's reputation",
    "crime_synthetic_identity_fraud": "creating a wholly fabricated, non-existent identity using AI, not based on a real person",
    "crime_fake_persona": "creating a fake online persona or account using AI-generated content",
    "crime_sexual_exploitation_ncii": "non-consensual intimate imagery created using AI, such as deepfake pornography",
    "crime_hate_speech_defamation": "attributing hateful or defamatory statements to someone using AI-manipulated content",
    "crime_authority_impersonation": "using AI to impersonate an authority figure or real person, such as in phishing or scam content",
}

HYPOTHESIS_TEMPLATE = "This post describes {}"  # won the template comparison (mean kappa 0.278 vs 0.190)

# Final calibrated thresholds per category, from the n=63 Gold Standard validation.
# modality_multimodal is deliberately set high (0.95): the model showed no reliable
# ranking signal for this category (the 2 true positives scored LOWER than dozens of
# true negatives), so this threshold defaults to "not multimodal" rather than flooding
# false positives -- see Known_Limitations in DeBERTa_Validation_Report.xlsx.
CALIBRATED_THRESHOLDS = {
    "modality_image": 0.05, "modality_video": 0.35, "modality_audio": 0.30,
    "modality_text": 0.15, "modality_multimodal": 0.95,
    "crime_biometric_bypass": 0.05, "crime_financial_fraud_ceo": 0.05,
    "crime_credential_theft": 0.80, "crime_document_forgery": 0.90,
    "crime_political_disinfo": 0.75, "crime_reputational_sabotage": 0.40,
    "crime_synthetic_identity_fraud": 0.25, "crime_fake_persona": 0.45,
    "crime_sexual_exploitation_ncii": 0.20, "crime_hate_speech_defamation": 0.35,
    "crime_authority_impersonation": 0.15,
}

# For a corpus of thousands of posts, checkpoint more frequently in wall-clock terms
# but less frequently in row count, to balance I/O overhead vs. Colab-disconnect risk.
CHECKPOINT_EVERY = 50
CHECKPOINT_CSV = "deberta_fullcorpus_checkpoint.csv"
OUTPUT_XLSX = "deberta_fullcorpus_results.xlsx"


In [ ]:
# %% CELL 4 — Load model on GPU
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"Using {'GPU' if device == 0 else 'CPU'} (device={device})")
if device == -1:
    print("WARNING: No GPU detected. Go to Laufzeit -> Laufzeittyp aendern -> GPU before "
          "running on the full corpus -- this will be extremely slow on CPU for thousands of posts.")

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=device,
)

Using GPU (device=0)


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [ ]:
# %% CELL 5 — Classification function
def classify_post(text):
    if not text or str(text).strip() == "":
        return {k: {"score": 0.0, "pred": 0} for k in CANDIDATE_LABELS}
    labels = list(CANDIDATE_LABELS.values())
    label_to_key = {v: k for k, v in CANDIDATE_LABELS.items()}
    result = classifier(
        str(text),
        candidate_labels=labels,
        hypothesis_template=HYPOTHESIS_TEMPLATE,
        multi_label=True,
    )
    out = {}
    for label, score in zip(result["labels"], result["scores"]):
        key = label_to_key[label]
        out[key] = {"score": round(score, 4), "pred": int(score >= CALIBRATED_THRESHOLDS[key])}
    return out

In [ ]:
# %% CELL 6 — Batch run with checkpointing (CRITICAL for a corpus this size)
import pandas as pd
import os
import time

def run_batch_full_corpus(input_csv, checkpoint_csv, id_col):
    df = pd.read_csv(input_csv)
    print(f"Total posts in corpus: {len(df)}")

    if os.path.exists(checkpoint_csv):
        done_df = pd.read_csv(checkpoint_csv)
        done_ids = set(done_df[id_col])
        print(f"Resuming from checkpoint: {len(done_ids)} posts already done.")
    else:
        done_df = pd.DataFrame()
        done_ids = set()

    todo = df[~df[id_col].isin(done_ids)]
    print(f"{len(todo)} posts remaining to classify.")

    results = []
    start_time = time.time()
    for i, (_, row) in enumerate(todo.iterrows(), start=1):
        text = f"{row.get('title', '')}. {row.get('selftext', '')}"
        preds = classify_post(text)
        rec = {id_col: row[id_col]}
        for cat, res in preds.items():
            rec[f"{cat}_score"] = res["score"]
            rec[f"{cat}_pred"] = res["pred"]
        results.append(rec)

        if i % CHECKPOINT_EVERY == 0 or i == len(todo):
            partial_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            partial_df.to_csv(checkpoint_csv, index=False)
            elapsed = time.time() - start_time
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (len(todo) - i) / rate if rate > 0 else float('inf')
            print(f"Checkpoint: {len(partial_df)}/{len(df)} done "
                  f"({time.strftime('%H:%M:%S')}, ~{rate:.1f} posts/sec, "
                  f"~{remaining/60:.0f} min remaining)")

    return pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)


In [ ]:
# %% CELL 7 — RUN IT (this can take a while for thousands of posts -- that's expected)
# If Colab disconnects/times out, just re-run this cell: it resumes automatically
# from deberta_fullcorpus_checkpoint.csv (re-upload it first via files.upload() if
# you're in a fresh Colab session, then re-run CELL 2-6 setup, then this cell).
model_df = run_batch_full_corpus(INPUT_CSV, CHECKPOINT_CSV, ID_COLUMN)
print("\nDone. Preview:")
model_df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 2


In [ ]:
def run_batch_full_corpus(input_csv, checkpoint_csv, id_col):
    try:
        df = pd.read_csv(input_csv, encoding="utf-8")
    except UnicodeDecodeError:
        print("UTF-8 fehlgeschlagen, versuche latin1/cp1252...")
        df = pd.read_csv(input_csv, encoding="latin1")
    print(f"Total posts in corpus: {len(df)}")

    if os.path.exists(checkpoint_csv):
        done_df = pd.read_csv(checkpoint_csv)
        done_ids = set(done_df[id_col])
        print(f"Resuming from checkpoint: {len(done_ids)} posts already done.")
    else:
        done_df = pd.DataFrame()
        done_ids = set()

    todo = df[~df[id_col].isin(done_ids)]
    print(f"{len(todo)} posts remaining to classify.")

    results = []
    start_time = time.time()
    for i, (_, row) in enumerate(todo.iterrows(), start=1):
        text = f"{row.get('title', '')}. {row.get('selftext', '')}"
        preds = classify_post(text)
        rec = {id_col: row[id_col]}
        for cat, res in preds.items():
            rec[f"{cat}_score"] = res["score"]
            rec[f"{cat}_pred"] = res["pred"]
        results.append(rec)

        if i % CHECKPOINT_EVERY == 0 or i == len(todo):
            partial_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            partial_df.to_csv(checkpoint_csv, index=False)
            elapsed = time.time() - start_time
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (len(todo) - i) / rate if rate > 0 else float('inf')
            print(f"Checkpoint: {len(partial_df)}/{len(df)} done "
                  f"({time.strftime('%H:%M:%S')}, ~{rate:.1f} posts/sec, "
                  f"~{remaining/60:.0f} min remaining)")

    return pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)

In [ ]:
def load_corpus_csv(input_csv):
    for enc in ["utf-8", "latin1"]:
        try:
            df = pd.read_csv(input_csv, encoding=enc, sep=None, engine="python")
            print(f"Loaded successfully with encoding={enc}, auto-detected separator.")
            return df
        except Exception as e:
            print(f"Failed with encoding={enc}: {type(e).__name__}")
    # Last resort: skip broken rows rather than crash entirely
    print("Falling back to utf-8 with bad lines skipped (some rows may be lost).")
    df = pd.read_csv(input_csv, encoding="utf-8", sep=None, engine="python", on_bad_lines="skip")
    return df


def run_batch_full_corpus(input_csv, checkpoint_csv, id_col):
    df = load_corpus_csv(input_csv)
    print(f"Total posts in corpus: {len(df)}")

    if os.path.exists(checkpoint_csv):
        done_df = pd.read_csv(checkpoint_csv)
        done_ids = set(done_df[id_col])
        print(f"Resuming from checkpoint: {len(done_ids)} posts already done.")
    else:
        done_df = pd.DataFrame()
        done_ids = set()

    todo = df[~df[id_col].isin(done_ids)]
    print(f"{len(todo)} posts remaining to classify.")

    results = []
    start_time = time.time()
    for i, (_, row) in enumerate(todo.iterrows(), start=1):
        text = f"{row.get('title', '')}. {row.get('selftext', '')}"
        preds = classify_post(text)
        rec = {id_col: row[id_col]}
        for cat, res in preds.items():
            rec[f"{cat}_score"] = res["score"]
            rec[f"{cat}_pred"] = res["pred"]
        results.append(rec)

        if i % CHECKPOINT_EVERY == 0 or i == len(todo):
            partial_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            partial_df.to_csv(checkpoint_csv, index=False)
            elapsed = time.time() - start_time
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (len(todo) - i) / rate if rate > 0 else float('inf')
            print(f"Checkpoint: {len(partial_df)}/{len(df)} done "
                  f"({time.strftime('%H:%M:%S')}, ~{rate:.1f} posts/sec, "
                  f"~{remaining/60:.0f} min remaining)")

    return pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)

In [ ]:
test_df = load_corpus_csv(INPUT_CSV)
print(test_df.shape)
print(test_df.columns.tolist())
test_df.head()

Failed with encoding=utf-8: UnicodeDecodeError
Failed with encoding=latin1: ParserError
Falling back to utf-8 with bad lines skipped (some rows may be lost).


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x98 in position 16: invalid start byte

In [ ]:
from google.colab import files
uploaded = files.upload()  # select Final_Retrieval_148_RESULTS_v1_3.xlsx
INPUT_XLSX = list(uploaded.keys())[0]

Saving Final_Retrieval_148_RESULTS_v1_3.xlsx to Final_Retrieval_148_RESULTS_v1_3 (1).xlsx


In [ ]:
def load_corpus_excel(input_xlsx, sheet_name="Candidate Posts"):
    df = pd.read_excel(input_xlsx, sheet_name=sheet_name, engine="openpyxl")
    print(f"Loaded {len(df)} rows from sheet '{sheet_name}'")
    return df

def run_batch_full_corpus(input_xlsx, checkpoint_csv, id_col):
    df = load_corpus_excel(input_xlsx)
    print(f"Total posts in corpus: {len(df)}")

    if os.path.exists(checkpoint_csv):
        done_df = pd.read_csv(checkpoint_csv)
        done_ids = set(done_df[id_col])
        print(f"Resuming from checkpoint: {len(done_ids)} posts already done.")
    else:
        done_df = pd.DataFrame()
        done_ids = set()

    todo = df[~df[id_col].isin(done_ids)]
    print(f"{len(todo)} posts remaining to classify.")

    results = []
    start_time = time.time()
    for i, (_, row) in enumerate(todo.iterrows(), start=1):
        text = f"{row.get('title', '')}. {row.get('selftext', '')}"
        preds = classify_post(text)
        rec = {id_col: row[id_col]}
        for cat, res in preds.items():
            rec[f"{cat}_score"] = res["score"]
            rec[f"{cat}_pred"] = res["pred"]
        results.append(rec)

        if i % CHECKPOINT_EVERY == 0 or i == len(todo):
            partial_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            partial_df.to_csv(checkpoint_csv, index=False)
            elapsed = time.time() - start_time
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (len(todo) - i) / rate if rate > 0 else float('inf')
            print(f"Checkpoint: {len(partial_df)}/{len(df)} done "
                  f"({time.strftime('%H:%M:%S')}, ~{rate:.1f} posts/sec, "
                  f"~{remaining/60:.0f} min remaining)")

    return pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)

In [ ]:
test_df = load_corpus_excel(INPUT_XLSX)
print(test_df.shape)
print(test_df.columns.tolist())
test_df.head()

Loaded 3971 rows from sheet 'Candidate Posts'
(3971, 15)
['Subreddit', 'post_id', 'created_utc', 'title', 'selftext', 'url', 'score', 'num_comments', 'Trigger rule', 'DIRECT hits', 'Group B hits', 'Group C hits', 'Group D hits', 'Group E hits (logged only, non-decisive)', 'Retrieval status']


,Subreddit,post_id,created_utc,title,selftext,url,score,num_comments,Trigger rule,DIRECT hits,Group B hits,Group C hits,Group D hits,"Group E hits (logged only, non-decisive)",Retrieval status
0,AI4tech,1r682li,2026-02-16 12:15:15+00:00,AI-Driven Fraud Is Blurring Reality: Is Your T...,A new Forbes Tech Council report warns that ge...,https://www.forbes.com/councils/forbestechcoun...,3,0,DIRECT,deepfake,NaN,NaN,fraud | scam,NaN,CANDIDATE — content validation still required
1,AI4tech,1sn3fgz,2026-04-16 13:23:42+00:00,Which video face swap works better?,[removed],https://www.reddit.com/r/AI4tech/comments/1sn3...,1,0,DIRECT,video face swap,NaN,face swap | video face swap,NaN,NaN,CANDIDATE — content validation still required
2,AI4tech,1vpd5jc,2026-08-15 20:24:34+00:00,So lets talk about what happened to Nano Banan...,Idea was of course to implement ai gen capabil...,https://www.reddit.com/r/AI4tech/comments/1vpd...,2,0,DIRECT,deepfake,NaN,NaN,NaN,NaN,CANDIDATE — content validation still required
3,AIGuild,1mad7sp,2025-07-27 04:24:11+00:00,"America’s AI Playbook: Build Faster, Safer, an...",**TLDR** The White House released a national A...,https://www.reddit.com/r/AIGuild/comments/1mad...,1,0,DIRECT,deepfake,synthetic media,NaN,NaN,NaN,CANDIDATE — content validation still required
4,AIGuild,1mej5qs,2025-08-01 01:12:09+00:00,"AI, Copyright Wars, and Deep-Fake Danger: A La...",**TLDR** An intellectual-property professor ex...,https://www.reddit.com/r/AIGuild/comments/1mej...,1,0,DIRECT,deep fake,NaN,NaN,fake nudes | fraud,NaN,CANDIDATE — content validation still required


In [ ]:
model_df = run_batch_full_corpus(INPUT_XLSX, CHECKPOINT_CSV, ID_COLUMN)

Loaded 3971 rows from sheet 'Candidate Posts'
Total posts in corpus: 3971
3971 posts remaining to classify.


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Checkpoint: 50/3971 done (10:34:36, ~1.1 posts/sec, ~59 min remaining)
Checkpoint: 100/3971 done (10:35:19, ~1.1 posts/sec, ~57 min remaining)
Checkpoint: 150/3971 done (10:35:59, ~1.2 posts/sec, ~54 min remaining)
Checkpoint: 200/3971 done (10:36:42, ~1.2 posts/sec, ~54 min remaining)
Checkpoint: 250/3971 done (10:37:25, ~1.2 posts/sec, ~53 min remaining)
Checkpoint: 300/3971 done (10:38:09, ~1.2 posts/sec, ~53 min remaining)
Checkpoint: 350/3971 done (10:38:51, ~1.2 posts/sec, ~52 min remaining)
Checkpoint: 400/3971 done (10:39:34, ~1.2 posts/sec, ~51 min remaining)
Checkpoint: 450/3971 done (10:40:17, ~1.2 posts/sec, ~50 min remaining)
Checkpoint: 500/3971 done (10:40:59, ~1.2 posts/sec, ~50 min remaining)
Checkpoint: 550/3971 done (10:41:41, ~1.2 posts/sec, ~49 min remaining)
Checkpoint: 600/3971 done (10:42:23, ~1.2 posts/sec, ~48 min remaining)
Checkpoint: 650/3971 done (10:43:08, ~1.2 posts/sec, ~47 min remaining)
Checkpoint: 700/3971 done (10:43:50, ~1.2 posts/sec, ~47 min rema

In [ ]:
MODALITY_CATS = ["modality_image", "modality_video", "modality_audio", "modality_text", "modality_multimodal"]
CRIME_CATS = [c for c in CANDIDATE_LABELS if c.startswith("crime_")]

def derive_is_case(row):
    has_modality = any(row.get(f"{c}_pred", 0) == 1 for c in MODALITY_CATS)
    has_crime = any(row.get(f"{c}_pred", 0) == 1 for c in CRIME_CATS)
    return int(has_modality and has_crime)

model_df["is_case_model_derived"] = model_df.apply(derive_is_case, axis=1)
print(f"is_case_model_derived = 1 for {model_df['is_case_model_derived'].sum()} of {len(model_df)} posts "
      f"({model_df['is_case_model_derived'].mean()*100:.1f}%)")

is_case_model_derived = 1 for 2541 of 3971 posts (64.0%)


In [ ]:
OUTPUT_XLSX = "deberta_fullcorpus_results.xlsx"
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    model_df.to_excel(writer, sheet_name="DeBERTa_FullCorpus", index=False)
print(f"Saved: {OUTPUT_XLSX} ({len(model_df)} rows)")

Saved: deberta_fullcorpus_results.xlsx (3971 rows)


In [ ]:
files.download(OUTPUT_XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>